In [1]:
import tensorflow as tf
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

2024-12-27 18:16:51.668278: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-12-27 18:16:51.687489: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1735303611.699472   90373 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1735303611.703105   90373 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-12-27 18:16:51.715177: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

### Split data

In [2]:
iris = load_iris()

In [3]:
train_x, test_x, train_y, test_y = train_test_split(iris['data'], iris['target'], test_size=0.2, random_state=42)
train_x, valid_x, train_y, valid_y = train_test_split(train_x, train_y, test_size=0.25, random_state=42) 

In [4]:
print(train_x.shape, train_y.shape)
print(valid_x.shape, valid_y.shape)
print(test_x.shape, test_y.shape)
train_x = train_x.astype('float32')
valid_x = valid_x.astype('float32')
test_x = test_x.astype('float32')


(90, 4) (90,)
(30, 4) (30,)
(30, 4) (30,)


#### Perceptron in tensorflow

In [5]:
def create_perceptron(learning_rate=0.01):
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(1, activation='sigmoid', input_shape=(4,))
    ])

    model.compile(
        optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [6]:
def train_perceptron(model, train_x, train_y, valid_x, valid_y, epochs=100):
    history = model.fit(
        train_x, train_y,
        epochs=epochs,
        validation_data=(valid_x, valid_y),
        verbose=1
    )
    return history

In [7]:
def plot_training_history(history):
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title('Perceptron Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    # plot loss
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title('Binary Cross-Entropy Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    plt.show()

In [8]:
def evaluate_perceptron(model, X_test, y_test):
    loss, accuracy = model.evaluate(X_test, y_test)
    print(f"\nTest Loss: {loss:.4f}")
    print(f"Test Accuracy: {accuracy:.4f}")
    
    predictions = (model.predict(X_test) >= 0.5).astype(int)
    print("\nSample Predictions (First 10 instances):")
    for i in range(10):
        print(f"True: {y_test[i]}, Predicted: {predictions[i][0]}")
        
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(y_test, predictions)
    print("\nConfusion Matrix:")
    print(cm)

In [9]:
def plot_decision_boundary(model, X, y):
    plt.figure(figsize=(10, 6))
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
        np.arange(y_min, y_max, 0.02))
    
    Z = model.predict(np.c_[xx.ravel(), yy.ravel(),
        np.zeros_like(xx.ravel()),
        np.zeros_like(xx.ravel())])
    Z = (Z >= 0.5).astype(int)
    Z = Z.reshape(xx.shape)
    
    plt.contourf(xx, yy, Z, alpha=0.4)
    plt.scatter(X[:, 0], X[:, 1], c=y, alpha=0.8)
    plt.xlabel('Sepal Length (scaled)')
    plt.ylabel('Sepal Width (scaled)')
    plt.title('Perceptron Decision Boundary (First Two Features)')
    plt.show()

In [10]:
model = create_perceptron()
model.summary()
hist = train_perceptron(model, train_x, train_y, valid_x, valid_y)

/home/ouroboros/dev/global-venv-312/lib/python3.12/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1735303628.986888   90373 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4158 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9
W0000 00:00:1735303629.155131   90552 gpu_backend_lib.cc:579] Can't find libdevice directory ${CUDA_DIR}/nvvm/libdevice. This may result in compilation or runtime failures, if the program we try to run uses routines from libdevice.
Searched for CUDA in the following directories:
  ./cuda_sdk_lib
  ipykernel_launcher.runfiles/cuda_nvcc
  ipykern/cuda_nvcc
  
  /usr/local/cuda
  /home/ouroboros/de

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 1)              │             5 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5 (20.00 B)

 Trainable params: 5 (20.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100


I0000 00:00:1735303629.747185   90542 service.cc:148] XLA service 0x7bf118017590 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1735303629.747205   90542 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce RTX 4050 Laptop GPU, Compute Capability 8.9
2024-12-27 18:17:09.757418: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1735303629.803155   90542 cuda_dnn.cc:529] Loaded cuDNN version 90501
W0000 00:00:1735303629.864189   90542 gpu_backend_lib.cc:617] libdevice is required by this HLO module but was not found at ./libdevice.10.bc
2024-12-27 18:17:09.865559: W tensorflow/core/framework/op_kernel.cc:1841] OP_REQUIRES failed at xla_ops.cc:577 : INTERNAL: libdevice not found at ./libdevice.10.bc
2024-12-27 18:17:09.865593: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with 

InternalError: Graph execution error:

Detected at node StatefulPartitionedCall defined at (most recent call last):
  File "/usr/lib/python3.12/runpy.py", line 198, in _run_module_as_main

  File "/usr/lib/python3.12/runpy.py", line 88, in _run_code

  File "/home/ouroboros/dev/global-venv-312/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>

  File "/home/ouroboros/dev/global-venv-312/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance

  File "/home/ouroboros/dev/global-venv-312/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 739, in start

  File "/home/ouroboros/dev/global-venv-312/lib/python3.12/site-packages/tornado/platform/asyncio.py", line 205, in start

  File "/usr/lib/python3.12/asyncio/base_events.py", line 640, in run_forever

  File "/usr/lib/python3.12/asyncio/base_events.py", line 1992, in _run_once

  File "/usr/lib/python3.12/asyncio/events.py", line 88, in _run

  File "/home/ouroboros/dev/